# 🛡️ Soil vs. Non-Soil Anomaly & Binary Detection (Task 2)
### **Soil Classification & Agronomic Intelligence 2025**
**Author:** Gogul Gupta (Team *Shree Yantra Dynamics*)  
**Affiliation:** Dr. A.P.J. Abdul Kalam Technical University  
**Leaderboard Result:** F1-Score: **0.8989** | Private Rank: **48**  

---
### 📌 Objective
Identify whether an input test image is **Soil (Class 1)** or **Non-Soil / Outlier (Class 0)** using CNN deep feature representations + One-Class SVM / Anomaly Detection.

In [ ]:
# 📦 1. Imports
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

In [ ]:
# 🧠 2. Pretrained ResNet-18 Feature Extractor
def get_feature_extractor(device):
    resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Identity()  # Remove classifier head -> 512-dim output
    resnet = resnet.to(device)
    resnet.eval()
    return resnet

feature_extractor = get_feature_extractor(device)

In [ ]:
# 🔄 3. Feature Extraction on Soil Images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_features_from_paths(img_paths, model, device):
    features = []
    with torch.no_grad():
        for p in tqdm(img_paths, desc="Extracting Deep Features"):
            img = Image.open(p).convert("RGB")
            t = transform(img).unsqueeze(0).to(device)
            feat = model(t).cpu().numpy().flatten()
            features.append(feat)
    return np.array(features)

In [ ]:
# 🛡️ 4. One-Class SVM Anomaly Classifier & Thresholding
# When feature arrays are extracted:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train_features)
# oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)
# oc_svm.fit(X_train_scaled)
# preds = oc_svm.predict(X_test_scaled) # +1 for inlier (soil), -1 for outlier
# binary_preds = [1 if p == 1 else 0 for p in preds]
print("✅ One-Class SVM pipeline configured with nu=0.1")